# Setup Environment

## Only Run on First Set-Up

In [1]:
"""
%%bash
set -eo pipefail
ENV_NAME="rag-capstone" PLATFORM="linux-64" bash scripts/bootstrap_env.sh


mkdir -p "$CONDA_PREFIX/etc/conda/activate.d"
cat > "$CONDA_PREFIX/etc/conda/activate.d/java.sh" <<'EOF'

set -euo pipefail

export JAVA_HOME="${CONDA_PREFIX}"
export PATH="${CONDA_PREFIX}/bin:${PATH}"
EOF
chmod +x "$CONDA_PREFIX/etc/conda/activate.d/java.sh"
"""

'\n%%bash\nset -eo pipefail\nENV_NAME="rag-capstone" PLATFORM="linux-64" bash scripts/bootstrap_env.sh\n\n\nmkdir -p "$CONDA_PREFIX/etc/conda/activate.d"\ncat > "$CONDA_PREFIX/etc/conda/activate.d/java.sh" <<\'EOF\'\n\nset -euo pipefail\n\nexport JAVA_HOME="${CONDA_PREFIX}"\nexport PATH="${CONDA_PREFIX}/bin:${PATH}"\nEOF\nchmod +x "$CONDA_PREFIX/etc/conda/activate.d/java.sh"\n'

## Run each kernal restart

In [2]:
%%bash
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate rag-capstone

In [3]:
import os
import subprocess
from pathlib import Path

def _first_match(root: Path, name: str) -> str:
    for p in root.rglob(name):
        if p.is_file():
            return str(p)
    raise RuntimeError(f"Could not find {name} under {root}")

env_prefix = str(Path(os.sys.executable).resolve().parent.parent)

javac = _first_match(Path(env_prefix), "javac")
libjvm = _first_match(Path(env_prefix), "libjvm.so")

java_home = str(Path(libjvm).resolve().parent.parent.parent)

os.environ["CONDA_PREFIX"] = env_prefix
os.environ["JAVA_HOME"] = java_home
os.environ["JVM_PATH"] = libjvm
os.environ["PATH"] = f"{env_prefix}/bin:{java_home}/bin:" + os.environ.get("PATH", "")

print("Python:", os.sys.executable)
print("CONDA_PREFIX:", os.environ["CONDA_PREFIX"])
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("JVM_PATH:", os.environ["JVM_PATH"])
print("javac:", subprocess.check_output(["which", "javac"]).decode().strip())
print(subprocess.check_output(["javac", "-version"], stderr=subprocess.STDOUT).decode().strip())

Python: /home/sagemaker-user/.conda/envs/rag-capstone/bin/python
CONDA_PREFIX: /home/sagemaker-user/.conda/envs/rag-capstone
JAVA_HOME: /home/sagemaker-user/.conda/envs/rag-capstone/lib/jvm
JVM_PATH: /home/sagemaker-user/.conda/envs/rag-capstone/lib/jvm/lib/server/libjvm.so
javac: /home/sagemaker-user/.conda/envs/rag-capstone/bin/javac
javac 21.0.10-internal


In [4]:
%%capture
from src.utils.aws_secrets import bootstrap_env
bootstrap_env()

# Run Experiment

## Imports and Settings

In [5]:
import os
import logging
import json
import pickle
import litellm

from datetime import datetime

from src.utils.config import PipelineConfig
from src.pipeline import run_experiment
from src.utils.helpers import format_results_dataframe

/home/sagemaker-user/.conda/envs/rag-capstone/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
os.environ["LITELLM_LOG"] = "ERROR"
logging.getLogger("LiteLLM").setLevel(logging.ERROR)
logging.getLogger("litellm").setLevel(logging.ERROR)

## Load Data

In [7]:
queries_path = os.path.join(os.getcwd(), "hotpotqa", "hotpot_eval_300.json")
with open(queries_path, 'r', encoding='utf-8') as file:
        queries = json.load(file)

## Config

In [8]:
baseline_cfg = PipelineConfig(
    iterative=True,
    embedding_type="fused",
    top_k=25,
    k_sparse=50,
    k_dense=50,
    rrf_k=60,
    use_rerank=True,
    top_n=5,
    max_length=512,
    batch_size=32,
    temperature=0.0,
    max_tries=4,
    eval_temperature=0.0,
    eval_max_tokens=2048,
    log_full_prompts=False,
    max_rounds=3,
    max_plan_steps=6,
    max_contexts_final=None,
    step_top_k=5,
    use_mlflow=True,
    mlflow_artifact_dir="artifacts",
)

## Run and Save Experiment

In [9]:
raw_results = run_experiment(queries=queries, cfg=baseline_cfg)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
raw_results_path = os.path.join(os.getcwd(), "results", f"raw_results_{timestamp}.pkl")
with open(raw_results_path, 'wb') as file:
    pickle.dump(raw_results, file)

formatted_results = format_results_dataframe(examples=queries, results=raw_results)
formatted_results_path = os.path.join(os.getcwd(), "results", f"formatted_results_{timestamp}.pkl")
with open(formatted_results_path, 'wb') as file:
    pickle.dump(formatted_results, file)

  0%|          | 0/300 [00:00<?, ?it/s]

Attempting to initialize prebuilt index beir-v1.0.0-hotpotqa.splade-v3.
/home/sagemaker-user/pyserini_cache/indexes/lucene-inverted.beir-v1.0.0-hotpotqa.splade-v3.20250603.168a2d.55d5635f4af0979d880daa3e8a361440 already exists, skipping download.
Initializing beir-v1.0.0-hotpotqa.splade-v3...


Mar 15, 2026 6:37:02 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


Attempting to initialize prebuilt index beir-v1.0.0-hotpotqa.bge-base-en-v1.5.
/home/sagemaker-user/pyserini_cache/indexes/faiss-flat.beir-v1.0.0-hotpotqa.bge-base-en-v1.5.20240107.d2c08665e8cd750bd06ceb7d23897c94 already exists, skipping download.
Initializing beir-v1.0.0-hotpotqa.bge-base-en-v1.5...


  0%|          | 1/300 [00:54<4:31:27, 54.47s/it, last_s=54.41]

...

100%|█████████▉| 299/300 [3:33:31<01:12, 72.82s/it, last_s=144.41]

🏃 View run query-5a8ce15d554299653c1aa12f at: https://us-east-1.experiments.sagemaker.aws/#/experiments/3/runs/2998b25763d1400186314658f47e8463
🧪 View experiment at: https://us-east-1.experiments.sagemaker.aws/#/experiments/3


100%|██████████| 300/300 [3:34:03<00:00, 42.81s/it, last_s=32.21] 

🏃 View run query-5ae5be02554299546bf82f3e at: https://us-east-1.experiments.sagemaker.aws/#/experiments/3/runs/23ed2a70dd474340ac17d85373ba53a1
🧪 View experiment at: https://us-east-1.experiments.sagemaker.aws/#/experiments/3


🏃 View run batch-20260315-183701-97ac9842 at: https://us-east-1.experiments.sagemaker.aws/#/experiments/3/runs/35f5584f48da44cea64765f4f7b7ea74
🧪 View experiment at: https://us-east-1.experiments.sagemaker.aws/#/experiments/3
